In [2]:
import torch
import torchvision
from torch import nn
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt  # 用于绘图

plt.rcParams['font.sans-serif'] = ['SimHei'] 
plt.rcParams['axes.unicode_minus'] = False

In [3]:
class MyNet(nn.Module):
    def __init__(self):
        super(MyNet, self).__init__()
        self.model = nn.Sequential(
            nn.Conv2d(3, 32, 5, 1, 2),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 32, 5, 1, 2),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 5, 1, 2),
            nn.MaxPool2d(2),
            nn.Flatten(),
            nn.Linear(1024, 64),
            nn.ReLU(),
            nn.Linear(64, 10)
        )

    def forward(self, x):
        return self.model(x)

In [4]:
train_data = torchvision.datasets.CIFAR10("./dataset", train=True, transform=torchvision.transforms.ToTensor(), download=True)
test_data = torchvision.datasets.CIFAR10("./dataset", train=False, transform=torchvision.transforms.ToTensor(), download=True)

train_dataloader = DataLoader(train_data, batch_size=64)
test_dataloader = DataLoader(test_data, batch_size=64)

100.0%


Extracting ./dataset\cifar-10-python.tar.gz to ./dataset
Files already downloaded and verified


In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
mynet = MyNet().to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(mynet.parameters(), lr=0.01)

# 用于存储绘图数据
train_loss_list = []
test_acc_list = []

epoch = 10

for i in range(epoch):
    print(f"------ 第 {i+1} 轮训练开始 ------")
    
    # --- 训练阶段 ---
    mynet.train()
    running_loss = 0.0
    for data in train_dataloader:
        imgs, targets = data
        imgs, targets = imgs.to(device), targets.to(device)
        
        outputs = mynet(imgs)
        loss = loss_fn(outputs, targets)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()

    # 记录本轮平均训练 Loss
    avg_train_loss = running_loss / len(train_dataloader)
    train_loss_list.append(avg_train_loss)

    # --- 测试阶段 ---
    mynet.eval()
    total_accuracy = 0
    with torch.no_grad():
        for data in test_dataloader:
            imgs, targets = data
            imgs, targets = imgs.to(device), targets.to(device)
            outputs = mynet(imgs)
            accuracy = (outputs.argmax(1) == targets).sum()
            total_accuracy += accuracy

    final_acc = total_accuracy / len(test_data)
    test_acc_list.append(final_acc.cpu().item())
    
    print(f"第 {i+1} 轮结束, Loss: {avg_train_loss:.4f}, Accuracy: {final_acc:.2%}")

print("训练完成！")

------ 第 1 轮训练开始 ------
第 1 轮结束, Loss: 2.1962, Accuracy: 28.09%
------ 第 2 轮训练开始 ------
第 2 轮结束, Loss: 1.8474, Accuracy: 29.70%
------ 第 3 轮训练开始 ------
第 3 轮结束, Loss: 1.6589, Accuracy: 33.91%
------ 第 4 轮训练开始 ------
第 4 轮结束, Loss: 1.5399, Accuracy: 38.02%
------ 第 5 轮训练开始 ------
第 5 轮结束, Loss: 1.4622, Accuracy: 41.24%
------ 第 6 轮训练开始 ------
第 6 轮结束, Loss: 1.3965, Accuracy: 43.85%
------ 第 7 轮训练开始 ------
第 7 轮结束, Loss: 1.3353, Accuracy: 46.80%
------ 第 8 轮训练开始 ------
第 8 轮结束, Loss: 1.2768, Accuracy: 50.89%
------ 第 9 轮训练开始 ------
第 9 轮结束, Loss: 1.2222, Accuracy: 53.02%
------ 第 10 轮训练开始 ------
第 10 轮结束, Loss: 1.1708, Accuracy: 55.16%
训练完成！
